# GRS subcode in characteristic two

A complete toy recovery from a public generator matrix. We compute a holdout kernel,
recover a coherent local jet, and extract an equivalent support and multiplier.
The same recovered jet is finished by **both** quadratic bootstrapping/minor codes and direct sections.

This is a fixed, reproducible example, not a success-rate experiment. The secret is
created with the instance and used only in the final independent audit. No secret
support, curve, tangent, or jet is supplied to any recovery function.

The readable implementation is in [reference.py](reference.py). Run all cells with
a **SageMath kernel**, starting in this notebook directory. Results are saved under
`results/`.


In [1]:
from pathlib import Path
import json
import sys
from time import perf_counter
from sage.all import *
from sage.version import version as sage_version

HERE = Path.cwd()
assert (HERE / "reference.py").is_file(), "Start the kernel in keyrec-toy/"
sys.path.insert(0, str(HERE))
from reference import (
    generate_grs, generate_alternant, holdout_kernel, local_maps,
    separate_branches, continue_jet, bootstrap, minor_finish, direct_finish,
    equivalent_support, audit_key, audit_geometry, reject_tampered_key,
)
started = perf_counter()
print("SageMath", sage_version)


SageMath 10.7


## 1. Generate the instance

The generator returns a public matrix and a separate audit record. Recovery is
parameterized by the public field, ambient degree bound, and interpolation
parameters. The matrix is put in row-echelon form; the held-out column is chosen
outside its information set.


In [2]:
q, m, n, expected_k, D = 32, 1, 24, 6, 7
d, s, section_degree, seed = 3, 1, 2, 0
Y, secret_for_audit = generate_grs(q, n, expected_k, D, seed)
E = GF(q**m, 'a')
k = Y.nrows()
assert k == expected_k
position = next(j for j in range(n) if j not in Y.pivots())
print(f"Public code: [{n},{k}] over {Y.base_ring()}")
print(f"Ambient field: {E}; degree bound D={D}; held-out position={position}")
print("Extension-field modulus:", E.modulus())


Public code: [24,6] over Finite Field in a of size 2^5
Ambient field: Finite Field in a of size 2^5; degree bound D=7; held-out position=5
Extension-field modulus: x^5 + x^2 + 1


## 2. Compute the public holdout kernel

We impose all Hasse conditions of order less than $s$ at the retained columns.
The retained unit columns allow the monomial exponent bound $d-s$; the remaining
conditions are imposed explicitly. A kernel basis is checked against the entire
assembled matrix.

Here $s(n-1)>dD$, so the ordinary interpolation theorem proves that every returned
polynomial vanishes identically on the hidden curve. The held-out evaluations are
checked separately. We use a deterministic panel of at most 48 basis polynomials
for the remaining stages; its size is reported.


In [3]:
assert s*(n-1) > d*D
K_full, monomials, holdout_report = holdout_kernel(Y, position, d, s)
assert K_full.nrows() > 0 and holdout_report["heldout_values_zero"]
K = K_full[:min(48, K_full.nrows())]
holdout_report["polynomials_used"] = K.nrows()
holdout_report["ordinary_zero_count"] = int(s*(n-1))
holdout_report["composition_degree_bound"] = int(d*D)
print(holdout_report)


{'matrix_rows': 17, 'monomials': 50, 'kernel_dimension': 34, 'heldout_values_zero': True, 'seconds': 0.06938474997878075, 'polynomials_used': 34, 'ordinary_zero_count': 23, 'composition_degree_bound': 21}


## 3. Separate the branches at the held-out point

Let $J$ collect the gradients and $Q_h$ the quadratic Taylor terms. On
$\ker J/\langle y_i\rangle$, use the quadrics from the stationary subspace
$\mathcal H^0$. We check the sufficient ranks
$\operatorname{rank}J=k-m-1$ and $\dim\langle Q_h|_{\ker J}\rangle=\binom m2$.
An eigenvalue calculation in the quadratic quotient recovers the branches.
For $m=1$, the gradient kernel itself is the tangent plane.


In [4]:
J, Q, quadratic_monomials = local_maps(K, monomials, Y.column(position))
branches, stationary, branch_report = separate_branches(
    J, Q, quadratic_monomials, Y.column(position), E, m)
v1 = branches[0]
print(branch_report)


{'gradient_rank': 4, 'gradient_nullity': 2, 'quadratic_rank': 0, 'quotient_dimension': 1, 'quadratic_family': 'stationary forms', 'branches': 1}


## 4. Continue one coherent jet

At each order, solve the regular equations and the stationary equations one
order ahead. The fixed matrix $[J;L_0]$ is factored once, and its kernel must be
exactly the point/tangent plane. The recursion continues to the requested depth
even after the coefficient span fills the ambient vector space.

A fresh full substitution checks every selected holdout polynomial against the
completed jet. We recover order $\delta D-1$, sufficient for the direct sections
used below.


In [5]:
required_depth = section_degree*D - 1
jets, jet_report = continue_jet(
    K, monomials, J, Q, quadratic_monomials, stationary,
    Y.column(position), v1, required_depth)
assert jet_report["ambiguity_dimension"] == 2
assert jet_report["first_full_span_order"] < required_depth
print(jet_report)


{'continuation_rank': 4, 'ambiguity_dimension': 2, 'requested_depth': 13, 'recovered_depth': 13, 'first_full_span_order': 5, 'full_substitution': True, 'binary_normalization': False, 'seconds': 0.07933216681703925}


## 5. Finish through bootstrapping and the minor code

Use the prefix of order $r=\max(0,2D-n+1)$, together with the public columns,
to recover all quadratic equations of the chosen branch. Check gradient rank
$k-2$ at every column. The square roots of the $2\times2$ minors give a code
contained in a GRS code of dimension $D$ on the original support.

We require its rank to equal $D$, recover its support by Sidelnikov–Shestakov,
and verify **exact GRS row-space equality**. A separate multiplier solve embeds
the original public code in a GRS code of dimension $D+1$.

For the small GRS example, $n>2D$, so the public points alone already determine
the quadratic ideal. The alternant example has positive required jet depth and
uses the selected jet to align the branch.


In [6]:
boot = bootstrap(Y, E, jets, D)
minor = minor_finish(Y, E, boot, D, expected_dimension=D)
minor_public_check = audit_key(Y, minor, D)
assert reject_tampered_key(Y, minor, D)
print("Bootstrap:", boot["report"])
print("Minor finishing:", minor["report"])
print("Public verification:", minor_public_check)


Bootstrap: {'degree': 2, 'jet_order': 0, 'constraint_rank': 15, 'ideal_dimension': 6, 'gradient_ranks': [4], 'positions_verified': 24}
Minor finishing: {'multiplier_nullity': 1, 'public_grs_containment': True, 'minor_rows': 15, 'minor_rank': 7, 'exact_grs_equality': True, 'recovered_positions': 24}
Public verification: {'distinct_support': True, 'public_grs_containment': True}


## 6. Finish by direct extraction

Find degree-$\delta$ homogeneous forms $R,S$ whose restrictions to the hidden
curve vanish at the seed to orders $\delta D$ and $\delta D-1$.
The numerator is required to be nonzero at a public column, and the denominator
has a nonzero leading contact coefficient. These are checked linear conditions.

Their evaluation ratio is a single Möbius transformation of the support. The seed
is assigned value zero; a possible point at infinity is moved to a finite chart.
The multiplier is recovered by public linear equations.

These are generic characteristic-two codes, so adjacent contact orders and ordinary ratios apply. The square-root modification for binary Goppa codes is a different specialization.


In [7]:
# Use only the sufficient jet prefix through order delta*D-1.
direct = direct_finish(Y, E, jets[:required_depth+1], position, D,
                       form_degree=section_degree)
direct_public_check = audit_key(Y, direct, D)
assert reject_tampered_key(Y, direct, D)
print("Direct finishing:", direct["report"])
print("Public verification:", direct_public_check)
print("First eight recovered support values:", direct["support"][:8])


Direct finishing: {'multiplier_nullity': 1, 'public_grs_containment': True, 'form_degree': 2, 'jet_order': 13, 'numerator_space': 7, 'denominator_space': 8, 'square_root': False, 'recovered_positions': 24, 'distinct_support': True}
Public verification: {'distinct_support': True, 'public_grs_containment': True}
First eight recovered support values: [0, a^3 + a + 1, a^3, a^4 + a^3 + 1, a, a^4 + a, a^4 + a^3, a + 1]


## 7. Independent audit using the generated secret

**Recovery has finished.** Only this stage uses the original support and
multiplier. It checks exact polynomial identities, all recovered branch planes,
and fits one scalar series and one invertible parameter series to every jet
coefficient. Thus coherence is checked at the coefficient level.

It also verifies that the recovered support differs from the generated support
by one global Frobenius power and one Möbius transformation.
The recovered quadratic ideal is compared with the full secret-curve ideal, and all tangent planes must belong to that same branch. The two finishing routes are compared directly as well.


In [8]:
geometry_check = audit_geometry(
    Y, secret_for_audit, K, monomials, position, jets, q,
    bootstrap_data=boot, branches=branches)
direct_secret_check = equivalent_support(
    secret_for_audit["support"], direct["support"], E, q)
print("Geometry:", geometry_check)
print("Direct support audit:", direct_secret_check)
minor_secret_check = equivalent_support(
    secret_for_audit["support"], minor["support"], E, q)
route_agreement = equivalent_support(minor["support"], direct["support"], E, E.order())
print("Minor support audit:", minor_secret_check)
print("Agreement between finishing routes:", route_agreement)


Geometry:

 {'exact_holdout_curve_identities': True, 'audited_polynomials': 34, 'coherent_jet': True, 'coherent_coordinate_coefficients': 84, 'audited_frobenius_branch': 0, 'all_branches_match_secret_audit': True, 'exact_full_branch_ideal': True, 'all_tangents_on_same_branch': True}
Direct support audit: {'frobenius_power': 0, 'mobius_equivalence': True}
Minor support audit: {'frobenius_power': 0, 'mobius_equivalence': True}
Agreement between finishing routes: {'frobenius_power': 0, 'mobius_equivalence': True}


## Recorded result

The full recovered keys remain in `direct` and, in characteristic two, `minor`.
This compact record describes exactly what was tested. Timings include instance
generation, public recovery, and the audits in this notebook; they are single-run
toy measurements, not a comparison of optimized implementations.


In [9]:
record = dict(
    notebook="01-grs-char2.ipynb", status="passed", sage_version=sage_version,
    family="grs", q=int(q), m=int(m), n=int(n), k=int(k), D=int(D),
    seed=int(seed), heldout_position=int(position), field_modulus=str(E.modulus()),
    recovery_inputs="public generator and public parameters only",
    supplied_secret_jets=False, supplied_secret_tangents=False,
    holdout=holdout_report, branches=branch_report, jet=jet_report,
    direct=direct["report"], direct_public_check=direct_public_check,
    direct_secret_check=direct_secret_check, geometry_audit=geometry_check,
    tampered_multiplier_rejected=True,
    total_seconds=perf_counter()-started)
record.update(bootstrap=boot["report"], minor=minor["report"],
              minor_public_check=minor_public_check, minor_secret_check=minor_secret_check,
              route_agreement=route_agreement)
record = json.loads(json.dumps(record, default=int))
(HERE / "results").mkdir(exist_ok=True)
(HERE / "results" / "01-grs-char2.json").write_text(json.dumps(record, indent=2) + "\n")
print("PASS — public recovery and independent audits completed")
print("Total seconds:", round(record["total_seconds"], 3))


PASS — public recovery and independent audits completed
Total seconds: 0.416
